# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Question shape:** yes/no with an observed label (`is_declining_label`). Per the
`training-honest-models` skill's table, that shape starts with **Logistic Regression, then
Random Forest** — readable first, stronger second, complexity only if it earns its place.

I'm training both, not picking one in advance, because the honest way to answer "does ML beat
the rule?" is to actually show both against the Week-4 baseline rather than assume the fancier
model wins. Gradient Boosting is on the menu but I'm skipping it here — this lane's dataset is
small (32 clients, 30k rows) and a depth-limited Random Forest is already close to as complex
as this amount of data safely supports; adding boosting on top wouldn't be earning complexity,
it'd be reaching for it.

**Features:** the same safe, non-leaky signals from earlier weeks — `search_volume`,
`competition`, `word_count`, `content_age_days`, `days_since_last_update`, `ctr`,
`avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impressions_90d`,
`sessions_90d`, `clicks_90d`. `trend_direction`/`trend_pct` are excluded — that's what the
label is built from.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42  # fixed seed, noted here so the table below reproduces on rerun

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric_features = ['search_volume', 'competition', 'word_count', 'content_age_days',
                     'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
                     'scroll_rate', 'ai_traffic_pct', 'impressions_90d', 'sessions_90d',
                     'clicks_90d']
numeric_features = [c for c in numeric_features if c in df.columns]
print(f"{len(numeric_features)} safe features:", numeric_features)
print(f"Overall base rate: {df['is_declining_label'].mean():.3f}")

13 safe features: ['search_volume', 'competition', 'word_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impressions_90d', 'sessions_90d', 'clicks_90d']
Overall base rate: 0.542


## 2. Split design

**Client-grouped holdout, not a random row split.** Every content item belongs to one of only
32 clients, and items from the same client share things a random split would leak across train
and test — domain authority, editorial style, industry seasonality. A random row-level split
would let the model partly memorize "this is a `client_X` page" rather than learn signals that
generalize to a client it has never seen. So I hold out entire clients (~20% of them) for
testing, matching the same `client_holdout` pattern the starter pipeline already uses. This is
a harder, more honest test than a random split — and it shows: the held-out clients' base rate
(39.1%) differs meaningfully from the training base rate (55.5%), which is exactly the kind of
distribution shift a random split would have hidden.

In [2]:
rng = np.random.default_rng(RANDOM_STATE)
clients = df['client_id'].dropna().unique()
shuffled = rng.permutation(clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = df['client_id'].isin(test_clients)

train_df, test_df = df[~test_mask].copy(), df[test_mask].copy()

print(f"Clients: {len(clients)} total, {n_test_clients} held out for test ({len(test_clients)} clients)")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")
print(f"Train base rate: {train_df['is_declining_label'].mean():.3f}   "
      f"Test base rate: {test_df['is_declining_label'].mean():.3f}")
print("\n(Base rates differ across the split -- a reminder this is a genuinely harder test")
print("than shuffling all 30,000 rows together would have been.)")

Clients: 32 total, 6 held out for test (6 clients)
Train rows: 27,675   Test rows: 2,325
Train base rate: 0.555   Test base rate: 0.391

(Base rates differ across the split -- a reminder this is a genuinely harder test
than shuffling all 30,000 rows together would have been.)


## 3. Train + compare vs my baseline

Same test rows, same metric (precision@K) as Week 4's baseline — the baseline is rebuilt here
from the identical formula so all three numbers come from one notebook run, not a copy-pasted
score from last week.

In [3]:
def percentile_rank(s): return s.rank(pct=True, method='average')
def normalize(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})
    top = frame.sort_values('score', ascending=False).head(min(k, len(frame)))
    return float(top['y'].mean()) if len(top) else 0.0

# --- rebuild the Week-4 baseline, same formula, evaluated on this same test slice ---
eligible = ((df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['impressions_90d'] >= 100)).astype(int)
df['visibility_score'] = percentile_rank(np.log1p(df['impressions_90d']))
df['position_quality_score'] = 1 - normalize(df['avg_position'].clip(lower=1, upper=20))
df['ctr_gap_score'] = 1 - percentile_rank(df['ctr'])
df['baseline_action_score'] = (eligible * (0.4*df['visibility_score']
                                            + 0.3*df['position_quality_score']
                                            + 0.3*df['ctr_gap_score'])).round(4)
test_df['baseline_action_score'] = df.loc[test_df.index, 'baseline_action_score']

def build_X(frame):
    X = frame[numeric_features].apply(pd.to_numeric, errors='coerce')
    return X.replace([np.inf, -np.inf], np.nan).fillna(0)

X_train, X_test = build_X(train_df), build_X(test_df)
y_train, y_test = train_df['is_declining_label'], test_df['is_declining_label']

logreg = Pipeline([('scaler', StandardScaler()),
                    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                             random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
test_df['rf_score'] = rf_scores

K1, K2 = 50, int(len(test_df) * 0.20)
comparison = pd.DataFrame({
    'method': ['Baseline (rule)', 'Logistic Regression', 'Random Forest'],
    'precision@50': [
        precision_at_k(y_test, test_df['baseline_action_score'], K1),
        precision_at_k(y_test, logreg_scores, K1),
        precision_at_k(y_test, rf_scores, K1),
    ],
    'precision@20%': [
        precision_at_k(y_test, test_df['baseline_action_score'], K2),
        precision_at_k(y_test, logreg_scores, K2),
        precision_at_k(y_test, rf_scores, K2),
    ],
})
print(f"K1=50, K2 (top 20% of {len(test_df)} test rows)={K2}, test base rate={y_test.mean():.3f}\n")
comparison.round(3)

K1=50, K2 (top 20% of 2325 test rows)=465, test base rate=0.391



,method,precision@50,precision@20%
0,Baseline (rule),0.46,0.555
1,Logistic Regression,0.22,0.559
2,Random Forest,0.70,0.609


## 4. Errors and interpretation

**The comparison table, read honestly:** Random Forest wins clearly on both cuts
(precision@50 ≈ 0.70, precision@20% ≈ 0.61, vs. the rule's 0.46 / 0.56). Logistic Regression is
the more interesting result — it actually **loses to the baseline at precision@50** (≈0.22)
while roughly tying it at precision@20% (≈0.56). That's not a bug to hide; per the skill, "if
the model wins at one K and loses at another, report both — that IS the finding." A linear
model apparently can't isolate the very top of the queue as well as the hand-written rule can,
even though it's roughly competitive once the queue widens to 20%.

**What the Random Forest leans on:** `impressions_90d` and `avg_position` dominate feature
importance, followed by `content_age_days`. That's sane — those are exactly the visibility and
ranking signals the whole lane is built around, not a suspiciously perfect single feature that
would suggest hidden leakage.

**Where it's wrong, concretely:**
- **False positives (top 50, not actually declining):** 15 of the top 50 flagged pages are not
  currently declining. These tend to be mid-position pages (avg_position ~10-26) with modest
  but real impressions and above-average word count (2,400-3,000+ words) — pages that look
  risk-worthy on paper but happen to be holding steady right now.
- **Missed decliners (true label=1, lowest model scores):** the model's biggest misses are
  pages with **near-zero impressions** (1-3 in the last 90 days). These are declining by the
  label's definition (a relative drop), but because `impressions_90d` is the top feature, a
  page with almost no traffic to begin with scores as low-priority regardless of its relative
  trend — the model is implicitly learning "low volume = don't bother," which quietly
  contradicts a small but real slice of the label it's supposed to predict.

In [4]:
top50 = test_df.sort_values('rf_score', ascending=False).head(50)
false_positives = top50[top50['is_declining_label'] == 0]
print(f"False positives in top 50: {len(false_positives)}/50")
print(false_positives[['content_id', 'rf_score', 'impressions_90d', 'avg_position', 'word_count']]
      .head(3).to_string(index=False))

print()
missed = test_df[test_df['is_declining_label'] == 1].sort_values('rf_score').head(3)
print("Missed decliners (true label=1, lowest model score):")
print(missed[['content_id', 'rf_score', 'impressions_90d', 'avg_position', 'word_count']].to_string(index=False))

print()
importances = pd.Series(rf.feature_importances_, index=numeric_features).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances.round(3))

False positives in top 50: 15/50
          content_id  rf_score  impressions_90d  avg_position  word_count
content_a1dd3f309e08  0.746309             6250          13.3      3024.0
content_00603b0349b4  0.744518             1076          25.6      2439.0
content_f8aa00130a0b  0.740308             1103           9.6      2689.0

Missed decliners (true label=1, lowest model score):
          content_id  rf_score  impressions_90d  avg_position  word_count
content_28b4223f4e5f  0.094716                1           0.0      3109.0
content_34b14c00f80c  0.102222                3           0.0       659.0
content_472ce7ae14c0  0.185780                3           0.3       684.0

Random Forest feature importances:
impressions_90d           0.265
avg_position              0.223
content_age_days          0.157
word_count                0.079
days_since_last_update    0.051
clicks_90d                0.049
scroll_rate               0.046
ctr                       0.042
sessions_90d              0.0